# Label distributions and dataset statistics

Generates the dataset statistics and the label-frequency figure (`latex/img/label_distribution.pdf`) used in the project proposal.

* SemEval-2023 Task 3 (subtask 3, EN and PL): train + dev pooled, because test gold labels are not public.
* MIPD (manipulation task): train + validation + test.

Run from the repository root or from `notebooks/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data"
IMG = ROOT / "latex" / "img"
IMG.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_semeval(lang: str) -> pd.DataFrame:
    def read_tsv(path: Path, col: str) -> pd.DataFrame:
        # Paragraph texts may contain tabs, so split only on the first two.
        with open(path, encoding="utf-8") as f:
            rows = [line.rstrip("\n").split("\t", 2) for line in f if line.strip()]
        return pd.DataFrame([r + [""] * (3 - len(r)) for r in rows], columns=["doc", "par", col])

    frames = []
    for split in ["train", "dev"]:
        base = DATA / "sem_eval" / lang / f"{split}-labels-subtask-3"
        df = read_tsv(Path(f"{base}.txt"), "labels").merge(
            read_tsv(Path(f"{base}.template"), "text"), on=["doc", "par"], how="inner"
        )
        df["split"] = split
        frames.append(df)
    df = pd.concat(frames, ignore_index=True)
    df["labels"] = df["labels"].apply(lambda s: [x for x in s.split(",") if x])
    return df


semeval = {lang: load_semeval(lang) for lang in ["en", "pl"]}
label_defs = pd.read_csv(DATA / "sem_eval" / "task3_def.csv", header=None, names=["label", "definition"])
SEMEVAL_LABELS = label_defs["label"].tolist()

MIPD_LABELS = {
    "CHERRY_PICKING": "Cherry Picking", "QUOTE_MINING": "Quote Mining", "ANECDOTE": "Anecdote",
    "WHATABOUTISM": "Whataboutism", "STRAWMAN": "Strawman", "LEADING_QUESTIONS": "Leading Questions",
    "EMOTIONAL_CONTENT": "Appeal to Emotion", "FALSE_CAUSE": "False Cause", "EXAGGERATION": "Exaggeration",
    "REFERENCE_ERROR": "Reference Error", "MISLEADING_CLICKBAI": "Misleading Clickbait",
}
mipd = pd.concat(
    [pd.read_csv(DATA / "mipd" / "manipulation" / f"{s}.csv").assign(split=s) for s in ["train", "validation", "test"]],
    ignore_index=True,
)
mipd[list(MIPD_LABELS)] = mipd[list(MIPD_LABELS)].astype(bool)

In [ ]:
def words(texts: pd.Series) -> pd.Series:
    return texts.str.split().str.len()


rows = []
for lang, df in semeval.items():
    n_labels = df["labels"].str.len()
    rows.append({
        "dataset": f"SemEval-2023 T3 ({lang.upper()})",
        "unit": "paragraph",
        "documents": df["doc"].nunique(),
        "items": len(df),
        "labels_present": len({l for ls in df["labels"] for l in ls}),
        "median_words": words(df["text"]).median(),
        "pct_no_label": 100 * (n_labels == 0).mean(),
        "mean_labels_if_any": n_labels[n_labels > 0].mean(),
    })
n_labels = mipd[list(MIPD_LABELS)].sum(axis=1)
rows.append({
    "dataset": "MIPD (manipulation)",
    "unit": "article",
    "documents": len(mipd),
    "items": len(mipd),
    "labels_present": int((mipd[list(MIPD_LABELS)].sum() > 0).sum()),
    "median_words": words(mipd["article"]).median(),
    "pct_no_label": 100 * (n_labels == 0).mean(),
    "mean_labels_if_any": n_labels[n_labels > 0].mean(),
})
stats = pd.DataFrame(rows).round(1)
stats

In [ ]:
per_split = pd.concat([
    *(df.groupby("split").agg(documents=("doc", "nunique"), items=("doc", "size")).assign(dataset=f"SemEval {lang.upper()}")
      for lang, df in semeval.items()),
    mipd.groupby("split").size().rename("items").to_frame().assign(documents=lambda d: d["items"], dataset="MIPD"),
]).reset_index()
per_split

In [ ]:
def semeval_freq(df: pd.DataFrame) -> pd.Series:
    counts = pd.Series([l for ls in df["labels"] for l in ls]).value_counts()
    return 100 * counts.reindex(SEMEVAL_LABELS, fill_value=0) / len(df)


sem_freq = pd.DataFrame({lang.upper(): semeval_freq(df) for lang, df in semeval.items()})
sem_freq = sem_freq.loc[sem_freq.max(axis=1).sort_values().index]
sem_freq.index = sem_freq.index.str.replace("_", " ")

mipd_freq = (100 * mipd[list(MIPD_LABELS)].mean()).rename(MIPD_LABELS).sort_values()

sem_freq.round(2), mipd_freq.round(2)

In [ ]:
plt.rcParams.update({"font.size": 7, "font.family": "serif", "axes.spines.top": False, "axes.spines.right": False})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.3, 3.0), gridspec_kw={"width_ratios": [1.15, 1]})

y = range(len(sem_freq))
h = 0.4
ax1.barh([i + h / 2 for i in y], sem_freq["EN"], height=h, color="#3b6ea5", label=f"EN ({stats.pct_no_label[0]:.0f}% no technique)")
ax1.barh([i - h / 2 for i in y], sem_freq["PL"], height=h, color="#d1495b", label=f"PL ({stats.pct_no_label[1]:.0f}% no technique)")
ax1.set_yticks(list(y), sem_freq.index)
ax1.set_xlabel("% of paragraphs")
ax1.set_title("SemEval-2023 Task 3 (train+dev)")
ax1.legend(frameon=False, loc="lower right")

y = range(len(mipd_freq))
ax2.barh(y, mipd_freq, height=0.65, color="#edae49")
ax2.set_yticks(list(y), mipd_freq.index)
ax2.set_xlabel("% of articles")
ax2.set_title("MIPD manipulation (all splits)")

fig.tight_layout()
fig.savefig(IMG / "label_distribution.pdf", bbox_inches="tight")
plt.show()